In [ ]:
# @title Gdrive 접근을 위한 drive mount

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## @title MP4 파일들 전부 M4a로 전환시키기

import os
import glob
import subprocess

folder_path = "/content/drive/MyDrive/2026-1"

# 3. 폴더 내의 모든 mp4 파일 목록 가져오기
mp4_files = glob.glob(f"{folder_path}/*.mp4")

if not mp4_files:
    print(f"❌ '{folder_path}' 경로에서 변환할 mp4 파일을 찾을 수 없습니다.")
else:
    print(f"📂 총 {len(mp4_files)}개의 mp4 파일을 찾았습니다.")
    print(f"저장 위치(동일 폴더): {folder_path}\n")

    # 4. 각 파일을 순회하며 ffmpeg로 변환 후 같은 폴더에 저장
    for file in mp4_files:
        # 원본 파일 전체 경로 (예: /content/drive/MyDrive/2026-1/0809_1.mp4)
        input_path = os.path.join(folder_path, file)

        # 파일 이름 분리 및 새 확장자 부여 (예: 0809_1.mp4 -> 0809_1.m4a)
        file_name_without_ext = os.path.splitext(file)[0]
        output_file = f"{file_name_without_ext}.m4a"

        # 결과물 전체 경로 (원본과 동일한 folder_path 사용)
        output_path = os.path.join(folder_path, output_file)

        print(f"🔄 변환 중: '{file}' ➔ '{output_file}'")

        # 실행할 ffmpeg 명령어 구성
        command = [
            "ffmpeg",
            "-i", input_path,
            "-vn",             # 비디오 제외
            "-c:a", "aac",     # 오디오 코덱 AAC
            "-y",              # 덮어쓰기 허용
            output_path
        ]

        # 명령어 실행
        subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print(f"\n✅ 모든 파일이 '{folder_path}'에 성공적으로 변환/저장되었습니다!")

In [ ]:
# @title Trasncript를 위한 whisper ai 설치

!pip install git+https://github.com/openai/whisper.git
!sudo apt update && sudo apt install ffmpeg
!pip install moviepy

In [ ]:
# @title M4a 파일들 transcript 만들기 (이미 만들어진 경우 제외)

import os
import glob

# 4. 경로 설정 및 파일 찾기
folder_path = "/content/drive/MyDrive/2026-1"
m4a_files = glob.glob(f"{folder_path}/*.m4a")

print(f"총 {len(m4a_files)}개의 m4a 파일을 찾았습니다.\n")

# 5. 작성하셨던 루핑 방지용 꼼꼼한 파라미터 설정
import whisper
model = whisper.load_model("medium")

#고정 prompt를 쓸 경우
fixed_prompt = "의학 강의 키워드: 항원, antigen, 항체, antibody, 면역글로불린, immunoglobulin, 사이토카인, 림프구, 대식세포, 염증, innate immunity, 선천면역, adaptive immunity, 적응면역, Hypersensitivity, Systemic sclerosis, 전신경화증, MCTD, 혼합결합조직질환, Rheumatoid Arthritis, 류마티스 관절염, Systemic Lupus Erythematosus, 전신홍반루푸스, Sjogren's Disease, 쇼그렌 증후군, Juvenile Idiopathic Arthritis, Inflammatory Myopathy, AOSD, Vasculitis, Behcet's disease, 베체트병, 아나필락시스, Anaphylaxis, Transplantation immunology, 이식면역, microbiome, 마이크로바이옴"

# 6. 반복문으로 변환 및 파일 저장
for file in m4a_files:
    base_name = os.path.splitext(os.path.basename(file))[0]
    save_path = os.path.join(folder_path, f"{base_name}_음성스크립트.txt")
    key_path = os.path.join(folder_path, f"{base_name}_whisperkeyword.txt")

    print(f"========== 작업 시작: {os.path.basename(file)} ==========")

    # 이미 파일이 있으면 건너뛰기 (중단 후 재시작 시 유용)
    if os.path.exists(save_path):
        print(f"⏭️ 이미 변환된 파일이 있어 건너뜁니다: {base_name}_음성스크립트.txt\n")
        continue

    try:
      if os.path.exists(key_path):
        with open(key_path, 'r', encoding='utf-8') as f:
            prompt = f.read()
            print(f"키워드 있음! : {prompt}\n")
      else:
            print(f"키워드 없음! ㅜㅜ\n")
            continue
            prompt = fixed_prompt

      transcribe_args = {
          "language": "ko",
          "initial_prompt": prompt,
          "condition_on_previous_text": False,          # 환각/루프 전염 방지
          "no_speech_threshold": 0.6,
          "compression_ratio_threshold": 2.0,           # 루핑 조기 차단
          "logprob_threshold": -1.0,                    # 엉뚱한 예측 차단
          "temperature": (0.0, 0.1, 0.2, 0.4, 0.6, 0.8) # 재시도 온도 세밀 조정
      }

      # 설정한 파라미터 적용하여 텍스트 추출
      result = model.transcribe(file, **transcribe_args)
      script_text = result["text"].strip()

      # 텍스트 파일로 드라이브에 저장
      with open(save_path, 'w', encoding='utf-8') as f:
          f.write(script_text)

      print(f"✅ 저장 완료: {base_name}_음성스크립트.txt\n")

    except Exception as e:
        print(f"❌ 변환 실패 ({base_name}): {e}\n")

In [ ]:
# @title 켜놓고 잘 때 튕김 방지

"""
이 아래는 코랩 튕김 방지 자바스크립트 코드 (여기서 실행 안 됨)

// 설정: 작동할 총 시간 (분 단위) - 예: 3시간 = 180분
const durationMinutes = 180;
const intervalTime = 60000; // 1분 간격
let elapsedTime = 0;

let colabTimer = setInterval(function() {
    const targetButton = document.querySelector("colab-toolbar-button");

    if (targetButton) {
        targetButton.click();
        console.log(`[코랩 유지] 세션 클릭 완료 (${elapsedTime / 60000 + 1}분 경과)`);
    } else {
        console.log("[코랩 유지] 경고: 연결 버튼을 찾지 못했습니다.");
    }

    elapsedTime += intervalTime;

    // 설정 시간이 지나면 타이머 종료
    if (elapsedTime >= durationMinutes * 60000) {
        clearInterval(colabTimer);
        console.log("✅ [코랩 유지] 설정한 시간이 지나 스크립트를 안전하게 종료합니다.");
    }
}, intervalTime);

"""

#작업 완료 후 죽이기
from google.colab import runtime
import time
print("✅ 특정 문구: 모든 작업이 완료되었습니다! 5초 뒤 코랩 세션을 종료합니다.")
time.sleep(5)
runtime.unassign()